# 03 — prioritizr optimization

Run the conservation optimization in **R** on the aligned hand-off stack and write results
back for Python (`04`). Reads the **manifest** contract written by `02`
(`input_data/aligned_stack/manifest.json`) and validates the grid before optimizing.

**Formulation (iteration 1, see `CLAUDE.md` / project memory):**

- **Objective** = `config.OBJECTIVE`. Default **min-shortfall with 100% targets** within an
  **area budget = 30% of the region** (30x30): this maximizes the captured *fraction* of
  every input, balanced across inputs (each on a 0–100% scale). Alternatives:
  `max_utility` (maximize total value; no floor) and `min_set` (minimize area to meet
  TARGET_PCT of every input; ignores the budget).
- **Protected areas locked in** and counted toward the budget.
- **EFG down-weighting** so the 40 EFGs do not swamp the 9 continuous features.
- **Connectivity penalty** (from the transboundary-connectivity surface) for contiguity —
  favors corridors linking PAs over scattered pixels. Off by default (magnitude is
  scale-dependent); calibrate via `CONNECTIVITY_PENALTY` in `config.py`.
- **MGA gap-portfolio** (`add_gap_portfolio`): a suite of structurally diverse
  near-optimal alternatives. **Requires Gurobi** (free academic license).

**Outputs** -> `output_data/iter1_minshortfall30/`: `portfolio.tif` (one alternative per
band), `selection_frequency.tif` (robustness/priority), `portfolio_representation.csv`
(feeds the 04 radar plot), `run_summary.json`.

**Kernel:** select **`R (y2y)`**. Run cell-by-cell; Ethan runs, Claude never executes.

In [1]:
# ---- Setup: libraries, paths, run parameters (from the manifest) ----------
suppressPackageStartupMessages({
  library(prioritizr)
  library(terra)
  library(jsonlite)
})

PROJ <- normalizePath(getwd())                 # run from the project root
MANIFEST <- file.path(PROJ, "input_data", "aligned_stack", "manifest.json")
stopifnot("manifest.json not found -- run the final cell of 02 first" = file.exists(MANIFEST))

manifest <- jsonlite::read_json(MANIFEST, simplifyVector = TRUE)
params <- manifest$params; grid <- manifest$grid; layers <- manifest$layers

# Run parameters: single source of truth is config.py, carried via the manifest.
SOLVER        <- params$solver                 # "highs" (prototype) or "gurobi" (portfolio)
HIGHS_SOLVER  <- params$highs_solver           # HiGHS LP algorithm ("ipm"/"simplex"/"choose")
TIME_LIMIT    <- params$solver_time_limit
AGG_FACTOR    <- params$prototype_agg_factor   # >1 = coarsen for prototype
DECISION_TYPE <- params$decision_type          # "binary" or "proportion"
OBJECTIVE     <- params$objective              # "min_shortfall" | "max_utility" | "min_set"
BUDGET_PCT    <- params$budget_pct
TARGET_PCT    <- params$target_pct             # 1.0 = maximize captured fraction (min_shortfall)
NORM_TOTAL    <- params$norm_total             # per-feature total after normalization
OPT_GAP       <- params$opt_gap
PORTFOLIO_N   <- params$portfolio_n
PORTFOLIO_GAP <- params$portfolio_gap
CONN_PENALTY  <- params$connectivity_penalty
BOUND_PENALTY <- params$boundary_penalty
RUN_TAG <- params$results_subdir
OUT_DIR <- file.path(PROJ, params$results_dir, RUN_TAG)
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

USE_PORTFOLIO <- identical(SOLVER, "gurobi")   # gap-portfolio only with Gurobi
have_gurobi <- requireNamespace("gurobi", quietly = TRUE)

cat(sprintf("prioritizr %s | terra %s | solver=%s%s\n",
            packageVersion("prioritizr"), packageVersion("terra"), SOLVER,
            if (USE_PORTFOLIO) sprintf(" (gap-portfolio n=%d)", PORTFOLIO_N) else " (single solution)"))
cat(sprintf("objective=%s | budget=%.0f%% target=%.0f%% | opt_gap=%.2f | time_limit=%ss\n",
            OBJECTIVE, 100*BUDGET_PCT, 100*TARGET_PCT, OPT_GAP, TIME_LIMIT))
cat(sprintf("connectivity_penalty=%g boundary_penalty=%g\n", CONN_PENALTY, BOUND_PENALTY))
cat(sprintf("outputs -> %s\n", sub(paste0(PROJ, "/"), "", OUT_DIR)))

prioritizr 8.1.0 | terra 1.9.34 | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=1800s
connectivity_penalty=0 boundary_penalty=0.0001
outputs -> output_data/iter1_minshortfall30


In [2]:
# ---- Ingest the hand-off stack + validate the grid (the Python->R contract) ----
abspath <- function(p) file.path(PROJ, p)

is_cont <- layers$role == "feature_continuous"
is_efg  <- layers$role == "feature_efg"
feat_rows <- layers[is_cont | is_efg, ]          # continuous first, then EFG (manifest order)
cost_row  <- layers[layers$role == "cost", ]
pa_row    <- layers[layers$role == "mask_locked_in", ]

features <- terra::rast(abspath(feat_rows$path)); names(features) <- feat_rows$name
cost     <- terra::rast(abspath(cost_row$path));  names(cost) <- "cost"
pa       <- terra::rast(abspath(pa_row$path));    names(pa) <- "pa"

# Validate every layer shares the canonical grid declared in the manifest.
b <- grid$bounds                                 # [left, bottom, right, top]
chk <- function(r, nm, problems) {
  if (terra::ncol(r) != grid$width || terra::nrow(r) != grid$height)
    problems <- c(problems, sprintf("%s: dim %dx%d != %dx%d", nm,
                  terra::ncol(r), terra::nrow(r), grid$width, grid$height))
  e <- unname(as.vector(terra::ext(r)))          # xmin,xmax,ymin,ymax
  if (!isTRUE(all.equal(e, c(b[1], b[3], b[2], b[4]), tolerance = 1e-3)))
    problems <- c(problems, sprintf("%s: extent mismatch", nm))
  problems
}
problems <- character(0)
for (i in seq_len(terra::nlyr(features))) problems <- chk(features[[i]], names(features)[i], problems)
problems <- chk(cost, "cost", problems); problems <- chk(pa, "pa", problems)
if (length(problems)) stop("GRID VALIDATION FAILED:\n  ", paste(problems, collapse = "\n  "))
cat(sprintf("ingested %d features (%d continuous + %d EFG) + cost + PA mask | grid %d x %d @ %d m\n",
            terra::nlyr(features), sum(is_cont), sum(is_efg), grid$width, grid$height, grid$res_m))

# ---- Prototype coarsening (optional) ------------------------------------
# The full 1 km binary MILP (~1.27M cells) is too large for HiGHS to presolve in time.
# AGG_FACTOR > 1 aggregates to a coarser grid for a fast trial (continuous features and
# EFG amounts by mean; cost/PA by mean). Set AGG_FACTOR = 1 for native 1 km (Gurobi).
if (AGG_FACTOR > 1) {
  features <- terra::aggregate(features, AGG_FACTOR, "mean", na.rm = TRUE)
  cost     <- terra::aggregate(cost,     AGG_FACTOR, "mean", na.rm = TRUE)
  pa       <- terra::aggregate(pa,       AGG_FACTOR, "mean", na.rm = TRUE)
  names(features) <- feat_rows$name; names(cost) <- "cost"; names(pa) <- "pa"
  cat(sprintf("PROTOTYPE: aggregated x%d -> %d x %d @ %d m (%s cells)\n",
              AGG_FACTOR, terra::ncol(cost), terra::nrow(cost), grid$res_m * AGG_FACTOR,
              format(terra::global(!is.na(cost), "sum", na.rm = TRUE)[[1]], big.mark = ",")))
}

# ---- Normalize each feature to a common total (numerical conditioning) ----
# Scale every feature so its total = NORM_TOTAL. With a 100% target (TARGET_PCT=1.0) the
# target equals the full total, which must stay < 1e6 for prioritizr's presolve; NORM_TOTAL
# (1e5) keeps it safe with well-scaled coefficients. min_shortfall/min_set are scale-
# invariant, so this does NOT change the solution -- pure conditioning. EFG 1-vs-2 ratio kept.
fsum <- terra::global(features, "sum", na.rm = TRUE)[, 1]
fsum[!is.finite(fsum) | fsum == 0] <- 1
features <- features * (NORM_TOTAL / fsum)        # each feature now totals NORM_TOTAL
names(features) <- feat_rows$name
cat(sprintf("normalized %d features to total=%g each (scale-invariant conditioning)\n",
            terra::nlyr(features), NORM_TOTAL))

ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
PROTOTYPE: aggregated x2 -> 643 x 1656 @ 2000 m (330,481 cells)
normalized 48 features to total=100000 each (scale-invariant conditioning)


In [3]:
# ---- Planning units, lock-in, and a feasibility check --------------------
# Cost (=1 per cell, NA outside the corridor) defines the planning units (PUs).
n_pu <- terra::global(!is.na(cost), "sum", na.rm = TRUE)[[1]]
total_cost <- terra::global(cost, "sum", na.rm = TRUE)[[1]]   # == n_pu (uniform cost)
BUDGET <- BUDGET_PCT * total_cost                            # area budget in cost units

# Lock in existing PAs: pa==1 within the PU set (terra read the 255 sentinel as NA
# outside the PU). Restrict to PUs so a non-PU cell can never be locked in.
locked <- terra::mask(pa >= 0.5, cost)
n_locked <- terra::global(locked, "sum", na.rm = TRUE)[[1]]

cat(sprintf("planning units: %s cells | budget = %.0f%% = %s cells\n",
            format(n_pu, big.mark=","), 100*BUDGET_PCT, format(round(BUDGET), big.mark=",")))
cat(sprintf("locked-in PAs : %s cells (%.1f%% of region) -- %s\n",
            format(n_locked, big.mark=","), 100*n_locked/n_pu,
            if (n_locked <= BUDGET) "fits within budget" else "EXCEEDS BUDGET"))
if (n_locked > BUDGET)
  stop("locked-in PA area exceeds the budget -> infeasible; relax BUDGET_PCT or lock-in.")

planning units: 330,481 cells | budget = 30% = 99,144 cells
locked-in PAs : 50,709 cells (15.3% of region) -- fits within budget


In [4]:
# ---- Feature weights: keep 40 EFGs from swamping the 9 continuous features ----
# Each continuous feature gets weight 1; the EFG group shares a total weight of 1
# (each EFG = 1/n_efg), so EFGs collectively count like one extra theme, not 40.
# Order matches the `features` stack (continuous first, then EFG).
n_cont <- sum(is_cont); n_efg <- sum(is_efg)
weights <- c(rep(1.0, n_cont), rep(1.0 / n_efg, n_efg))
names(weights) <- names(features)
cat(sprintf("weights: %d continuous @ 1.0 ; %d EFG @ %.4f (EFG group total = 1.0)\n",
            n_cont, n_efg, 1.0 / n_efg))

weights: 8 continuous @ 1.0 ; 40 EFG @ 0.0250 (EFG group total = 1.0)


In [5]:
# ---- Spatial penalties: connectivity + boundary (anti-scatter / clustering) ----
# Both discourage scattered selections; magnitudes are scale-dependent (tune in config).
# Boundary is normalized to edge units so BOUNDARY_PENALTY reads as "shortfall-equivalent
# cost per exposed cell edge". Built only when the corresponding penalty is on.
cell_m <- grid$res_m * AGG_FACTOR
if (CONN_PENALTY > 0) {
  cm <- prioritizr::connectivity_matrix(cost, features[["transboundary_connectivity"]])
  cat(sprintf("connectivity matrix: %s edges | weight range [%.4g, %.4g]\n",
              format(length(cm@x), big.mark = ","), min(cm@x), max(cm@x)))
}
if (BOUND_PENALTY > 0) {
  bm <- prioritizr::boundary_matrix(cost)
  bm <- bm / cell_m                                  # metres -> edge units
  cat(sprintf("boundary matrix built (edge units) for compactness penalty\n"))
}
cat(sprintf("penalties -> connectivity=%g | boundary=%g  (0 = off)\n", CONN_PENALTY, BOUND_PENALTY))

boundary matrix built (edge units) for compactness penalty
penalties -> connectivity=0 | boundary=0.0001  (0 = off)


In [6]:
# ---- Build the conservation problem -------------------------------------
# Minimum-shortfall: within the area budget, minimise the weighted shortfall from the
# per-feature targets. Targets are soft; PAs are locked in. SOLVER picks the engine:
# HiGHS returns ONE solution (prototype, no license cap); Gurobi adds the MGA
# gap-portfolio (suite of near-optimal alternatives; needs an unlimited academic license).
p <- problem(cost, features)
if (identical(OBJECTIVE, "min_set")) {
  # Ignore the budget; minimise AREA needed to meet TARGET_PCT of every feature.
  p <- p |> add_min_set_objective() |> add_relative_targets(TARGET_PCT)
} else if (identical(OBJECTIVE, "max_utility")) {
  # Maximise total (weighted) captured value within the area budget.
  p <- p |> add_max_utility_objective(budget = BUDGET) |> add_feature_weights(weights)
} else {
  # min_shortfall: minimise weighted shortfall from targets within the budget.
  # TARGET_PCT = 1.0 -> maximise the captured FRACTION of every input (balanced).
  p <- p |> add_min_shortfall_objective(budget = BUDGET) |>
            add_relative_targets(TARGET_PCT) |> add_feature_weights(weights)
}
p <- p |> add_locked_in_constraints(locked)

# Decision type: binary = reserve (MILP); proportion = fractional allocation (LP, fast).
p <- if (identical(DECISION_TYPE, "proportion")) p |> add_proportion_decisions() else p |> add_binary_decisions()

if (CONN_PENALTY > 0)  p <- p |> add_connectivity_penalties(CONN_PENALTY, data = cm)
if (BOUND_PENALTY > 0) p <- p |> add_boundary_penalties(BOUND_PENALTY, data = bm)

n_threads <- parallel::detectCores()
if (USE_PORTFOLIO) {
  stopifnot("SOLVER='gurobi' but the gurobi R package is not installed (see requirements-R.txt)" = have_gurobi)
  p <- p |>
    add_gap_portfolio(number_solutions = PORTFOLIO_N, pool_gap = PORTFOLIO_GAP) |>
    add_gurobi_solver(gap = OPT_GAP, time_limit = TIME_LIMIT, threads = n_threads, verbose = TRUE)
} else {
  # Single-solution prototype with HiGHS -- `s` will have one layer.
  # IPM (interior point) handles the large sparse boundary-penalty LP far better than
  # the default dual simplex; same LP optimum. Set via config HIGHS_SOLVER.
  p <- p |> add_highs_solver(gap = OPT_GAP, time_limit = TIME_LIMIT, threads = n_threads,
                             verbose = TRUE, control = list(solver = HIGHS_SOLVER))
}

print(p)

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (330481 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 99144.3)
│├•penalties: 
││└•1:          boundary penalties (`penalty` = 0.0001, `edge_factor` = 0.5, `formulation` = "simple", …)
│├•features:
││├•targets:    relative targets (all equal to 1)
││└•weights:    continuous values (between 0.025 and 1)
│├•constraints: 
││└•1:          locked in constraints (50709 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      highs solver (`gap` = 0.1, `time_limit` = 1800, …)
# ℹ Use `summary(...)` to see complete formulation.



In [7]:
# ---- Solve (Ethan runs; heavy) ------------------------------------------
# Full 1 km / ~1.27M-PU MILP. HiGHS prototype = one solution; Gurobi = the portfolio.
# Capped by TIME_LIMIT (returns the best solution so far); raise OPT_GAP for speed.
timing <- system.time(s <- solve(p))
# `s` is a SpatRaster: one near-optimal alternative per layer (1 layer for HiGHS).
n_sol <- terra::nlyr(s)
names(s) <- sprintf("alt_%02d", seq_len(n_sol))
cat(sprintf("solved with %s: %d solution(s) in %.1f s\n", SOLVER, n_sol, timing[["elapsed"]]))

LP has 1298863 rows; 979936 cols; 7188951 nonzeros

Coefficient ranges:

  Matrix  [1e-06, 1e+05]

  Cost    [2e-04, 1e+00]

  Bound   [1e+00, 1e+00]

  RHS     [1e+05, 1e+05]


Presolving model

1098243 rows, 835560 cols, 6051268 nonzeros 1s

1084909 rows, 822174 cols, 2449536 nonzeros 3s

Presolve reductions: rows 1084909(-213954); columns 822174(-157762); nonzeros 2449536(-4739415) 

Solving the presolved LP

IPX model has 1084909 rows, 822174 columns and 2449536 nonzeros

Input
    Number of variables:                                822174
    Number of free variables:                           0
    Number of constraints:                              1084909
    Number of equality constraints:                     0
    Number of matrix entries:                           2449536

    Matrix range:                                       [1e+00, 1e+00]

    RHS range:                                          [5e+04, 5e+04]

    Objective range:                                    [3e-0

In [8]:
# ---- Per-alternative summaries + selection-frequency map ----------------
rep_rows <- list(); stat_rows <- list()
for (i in seq_len(n_sol)) {
  sol <- s[[i]]
  fr <- eval_feature_representation_summary(p, sol)   # feature, total_amount, relative_held, ...
  fr$alternative <- names(s)[i]
  rep_rows[[i]] <- fr
  n_sel <- terra::global(sol, "sum", na.rm = TRUE)[[1]]
  n_new <- terra::global(sol & (locked == 0), "sum", na.rm = TRUE)[[1]]
  stat_rows[[i]] <- data.frame(alternative = names(s)[i], n_selected = n_sel,
                               pct_region = 100 * n_sel / n_pu, n_added_beyond_pa = n_new)
}
representation <- do.call(rbind, rep_rows)
stats <- do.call(rbind, stat_rows)

# Selection frequency: how many alternatives select each cell (robustness / priority).
sel_freq <- sum(s); names(sel_freq) <- "selection_frequency"

print(stats)

  alternative n_selected pct_region n_added_beyond_pa
1      alt_01    99144.3         30             48258


In [9]:
# ---- Write outputs: clean hand-off back to Python (04) -------------------
portfolio_tif <- file.path(OUT_DIR, "portfolio.tif")
freq_tif      <- file.path(OUT_DIR, "selection_frequency.tif")
rep_csv       <- file.path(OUT_DIR, "portfolio_representation.csv")
summary_json  <- file.path(OUT_DIR, "run_summary.json")

# Binary -> uint8 (255=NA); proportion -> float32 (NaN=NA).
if (identical(DECISION_TYPE, "proportion")) {
  terra::writeRaster(s, portfolio_tif, overwrite = TRUE, datatype = "FLT4S",
                     gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  terra::writeRaster(sel_freq, freq_tif, overwrite = TRUE, datatype = "FLT4S",
                     gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
} else {
  terra::writeRaster(s, portfolio_tif, overwrite = TRUE, datatype = "INT1U",
                     NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  terra::writeRaster(sel_freq, freq_tif, overwrite = TRUE, datatype = "INT1U",
                     NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
}
write.csv(representation, rep_csv, row.names = FALSE)

run_summary <- list(
  run_tag = RUN_TAG, objective = OBJECTIVE, params = params,
  n_planning_units = n_pu, budget_cells = round(BUDGET), n_locked_in = n_locked,
  n_alternatives = n_sol, solve_seconds = unname(timing[["elapsed"]]),
  per_alternative = stats,
  versions = list(R = as.character(getRversion()),
                  prioritizr = as.character(packageVersion("prioritizr")),
                  terra = as.character(packageVersion("terra")),
                  gurobi = if (have_gurobi) as.character(packageVersion("gurobi")) else NA))
jsonlite::write_json(run_summary, summary_json, auto_unbox = TRUE, pretty = TRUE, na = "null")

cat("wrote:\n")
for (f in c(portfolio_tif, freq_tif, rep_csv, summary_json))
  cat(sprintf("  %s\n", sub(paste0(PROJ, "/"), "", f)))

wrote:
  output_data/iter1_minshortfall30/portfolio.tif
  output_data/iter1_minshortfall30/selection_frequency.tif
  output_data/iter1_minshortfall30/portfolio_representation.csv
  output_data/iter1_minshortfall30/run_summary.json
